# Prepping Data Challenge - Week 22

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
loyalty_points_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_22_Loyalty_Points_Extended.xlsx', sheet_name='Loyalty Points')
store_data_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_22_Loyalty_Points_Extended.xlsx', sheet_name='Store Data')
customer_details_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_22_Loyalty_Points_Extended.xlsx', sheet_name='Customer Details')

In [3]:
loyalty_points_df.head()

,RecordID,Email Address,Loyalty Points,Store ID,DateTime_Out,Right_RecordID
0,1,Aarika.d@wikispaces.com,/__ 1.2 \LotaltyScore,6,01-Dec-2023,1
1,2,Abbey.m@tiny.cc,/__ 0.2 \LotaltyScore,6,30-Mar-2023,2
2,3,Abbi.p@chronoengine.com,/__ 4.8 \LotaltyScore,8,28-Mar-2023,3
3,4,Abby.l@desdev.cn,/__ 2.1 \LotaltyScore,2,10-Apr-2023,4
4,5,Abbye.h@t-online.de,/__ 0.4 \LotaltyScore,2,28-Oct-2023,5


In [4]:
store_data_df.head()

,City,Store,Store ID
0,London,Oxford Street,1
1,London,Wembley,2
2,London,Richmond,3
3,London,Stratford,4
4,Manchester,Trafford,5


In [5]:
customer_details_df.head()

,First Name,Last Name,Postcode,address
0,Aarika,Densham,NaN,Suite 52
1,Abbey,Moorey,NaN,12th Floor
2,Abbi,Parvin,NaN,PO Box 62649
3,Abby,Leonardi,6513,Suite 73
4,Abbye,Hazlehurst,NaN,8th Floor


## Challenge

**Goal:** Find the top 5 customers for each store, that do have a post code

### Feature Cleaning

In [6]:
# Convert Datetime to Datetime
loyalty_points_df['DateTime_Out'] = pd.to_datetime(loyalty_points_df['DateTime_Out'])

# Extract Loyalty Points as numeric
loyalty_points_df['Loyalty Points'] = loyalty_points_df['Loyalty Points'].apply(lambda x: float(x.split(' ')[1]))

In [7]:
loyalty_points_df.head()

,RecordID,Email Address,Loyalty Points,Store ID,DateTime_Out,Right_RecordID
0,1,Aarika.d@wikispaces.com,1.2,6,2023-12-01,1
1,2,Abbey.m@tiny.cc,0.2,6,2023-03-30,2
2,3,Abbi.p@chronoengine.com,4.8,8,2023-03-28,3
3,4,Abby.l@desdev.cn,2.1,2,2023-04-10,4
4,5,Abbye.h@t-online.de,0.4,2,2023-10-28,5


In [8]:
loyalty_points_df.shape

(999, 6)

### Joining Tables

In [9]:
# Extract information from Email Address for Join
loyalty_points_df['First Name'] = loyalty_points_df['Email Address'].apply(lambda x: x.split('.')[0])
loyalty_points_df['Last Name Initial'] = loyalty_points_df['Email Address'].apply(lambda x: x.split('.')[1].split('@')[0].upper())

In [10]:
# Extract initial of First Name from Customer DF
customer_details_df['Last Name Initial'] = customer_details_df['Last Name'].apply(lambda x: x[0].upper())

In [11]:
# Join all 3 DataFrames
joined_df = loyalty_points_df.merge(customer_details_df, on=['First Name', 'Last Name Initial'])
joined_df = joined_df.merge(store_data_df, on='Store ID')

In [12]:
# Remove unnecessary Fields
joined_df = joined_df.drop(columns=['Last Name Initial', 'Right_RecordID'])

In [13]:
joined_df.head()

,RecordID,Email Address,Loyalty Points,Store ID,DateTime_Out,First Name,Last Name,Postcode,address,City,Store
0,1,Aarika.d@wikispaces.com,1.2,6,2023-12-01,Aarika,Densham,NaN,Suite 52,Manchester,Salford
1,2,Abbey.m@tiny.cc,0.2,6,2023-03-30,Abbey,Moorey,NaN,12th Floor,Manchester,Salford
2,3,Abbi.p@chronoengine.com,4.8,8,2023-03-28,Abbi,Parvin,NaN,PO Box 62649,Birmingham,Birmingham
3,4,Abby.l@desdev.cn,2.1,2,2023-04-10,Abby,Leonardi,6513,Suite 73,London,Wembley
4,5,Abbye.h@t-online.de,0.4,2,2023-10-28,Abbye,Hazlehurst,NaN,8th Floor,London,Wembley


In [14]:
joined_df.shape

(999, 11)

### Find the top customers with postcodes 

In [15]:
# Filter out customers without post codes
joined_df = joined_df[joined_df['Postcode'].notna()]

In [16]:
joined_df['Loyalty Rank'] = joined_df.groupby(['City', 'Store'])['Loyalty Points'].rank(method='min', ascending=False)

In [17]:
top_5_customers_df = joined_df[joined_df['Loyalty Rank'] <= 5].sort_values(['City', 'Store', 'Loyalty Rank'])

In [18]:
top_5_customers_df

,RecordID,Email Address,Loyalty Points,Store ID,DateTime_Out,First Name,Last Name,Postcode,address,City,Store,Loyalty Rank
43,44,Andros.k@cafepress.com,9.8,8,2023-05-02,Andros,Kopfer,55590,PO Box 8729,Birmingham,Birmingham,1.0
523,524,Karalee.m@addtoany.com,9.8,8,2023-11-20,Karalee,Marcq,5449,PO Box 93919,Birmingham,Birmingham,1.0
745,746,Ofilia.a@newyorker.com,9.6,8,2023-11-06,Ofilia,Aizikovich,4212,Room 1698,Birmingham,Birmingham,3.0
32,33,Alphonso.p@yahoo.co.jp,9.4,8,2023-09-30,Alphonso,Portam,3313,Suite 38,Birmingham,Birmingham,4.0
53,54,Archibald.w@prweb.com,9.2,8,2023-08-26,Archibald,Watkins,446017,Suite 90,Birmingham,Birmingham,5.0
348,349,Forbes.a@csmonitor.com,9.8,10,2023-04-24,Forbes,Allsopp,5616,14th Floor,Leeds,Leeds,1.0
881,882,Staffard.r@techcrunch.com,9.8,10,2023-03-06,Staffard,Ronan,17130,PO Box 30482,Leeds,Leeds,1.0
773,774,Peggy.b@vimeo.com,9.7,10,2023-04-13,Peggy,Braisted,641730,12th Floor,Leeds,Leeds,3.0
993,994,Zara.m@unblog.fr,9.6,10,2023-08-28,Zara,Marten,169660,Apt 119,Leeds,Leeds,4.0
18,19,Aldo.b@washingtonpost.com,9.4,10,2023-04-09,Aldo,Beetham,8710,Suite 15,Leeds,Leeds,5.0


## Save Data

In [19]:
top_5_customers_df.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/22_top_5_customers.csv')